In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np

import pybedtools
from pybedtools import BedTool


#For plotting
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

#For statistics
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools

import re
from Bio import SeqIO
import ast # for safe eveal, for parsing some of the data
import math
# --- Shared helpers: const.py lives in analyses/common ---
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'common')))

#import importlib
#importlib.reload(const)

import const #to reload use import(importlib) and then importlib.reload(const)
from const import pos_active_ctrl_color,neg_active_ctrl_color,highlight_color,custom_cmap
from const import set_equal_plot_limits
from const import plot_color_pallete
from const import custom_cmap_bolder
from const import FONT_SIZE_small
const.set_plot_style()
import matplotlib.ticker as mtick

# --- Working directory (EDIT): used for relative .bed/.csv I/O in some notebooks ---
WORK_DIR = '/home/labs/davidgo/Collaboration/backup/humanMPRA/scripts/produce_paper_figures'
os.chdir(WORK_DIR)

output_path = '/home/labs/davidgo/Collaboration/humanMPRA/paper/raw_plots/'

# 1. Load original libraries

In [ ]:
library = pd.read_excel(f'/home/labs/davidgo/Collaboration/humanMPRA_raw/sequences.xlsx', 
                    sheet_name = 'oligos',
                    usecols=lambda x: x not in ["derived_seq_with_adapters", "ancestral_seq_with_adapters"],
                     header=0)

# 2. Filter libraries to not include hh oligos

In [ ]:
print(f'rows before filtering for SCREEN: {len(library)}')
library = library[library['source'].str.contains('SCREEN')]
print(f'rows after filtering for SCREEN: {len(library)}')
print(f'oligo counts after filtering for SCREEN: {len(library)*2}')


# 3. Read L4 library

In [ ]:
library_L4 = pd.read_excel(
    "/home/labs/davidgo/Collaboration/humanMPRA_raw/L4/L4_v5.xlsx",
    sheet_name="test_oligos",
    usecols=lambda x: x not in ["derived_sequence_with_adapters", "ancestral_seq_with_adapters"],
    header=0
)

#change colnames to match library
library_L4 = library_L4.rename(columns={
    'name_derived_sequence': 'name_ancestral',
    'name_ancestral_sequence': 'name_derived'})


# 4. Filter L4 to only include relevant oligos 

In [ ]:
library_L4_filtered = library_L4.copy()
library_L4_filtered = library_L4_filtered[(library_L4_filtered['adapter']=='a1')&
                                          (library_L4_filtered['name'].str.contains('SCREEN'))&
                                          (library_L4_filtered['class'].isin(['CGI.fix','promoter.orientation.fix']))]
print(f'oligo counts in L4: {len(library_L4_filtered)*2}')

# 3. Filter oligos from original libraries that were fixed in L4a1.

In [ ]:
fixed_variants = library_L4_filtered['variants_genomic']
fixed_oligos = sum(library['variants_genomic'].isin(fixed_variants))
print(f'oligos fixed in L4a1 which I removed from the original library design:{fixed_oligos}')

filtered_library = library[~library['variants_genomic'].isin(fixed_variants)]


# 4. merge L4 with the rest of the libraries and remove duplicates

In [ ]:
# columns shared by both
common_cols = filtered_library.columns.intersection(library_L4_filtered.columns)

# concatenate, keeping only shared columns (aligned + same order)
final_oligos = pd.concat([library_L4_filtered[common_cols],filtered_library[common_cols]], ignore_index=True)

final_oligos = final_oligos.drop_duplicates(
    subset="variants_genomic",
    keep="first"
)

In [ ]:
all_variants = (
    final_oligos["variants_genomic"]
      .dropna()
      .str.split(";")
      .explode()
      .tolist()
)
len(set(all_variants))


# Error: Change the number of variants to unique!!!!! 

In [ ]:
oligo_pairs_count = len(final_oligos)
oligos_count = oligo_pairs_count*2
variants_count = sum(final_oligos['variants_count'])

print(f'final number of oligo pairs: {oligo_pairs_count}')
print(f'final number of oligos: {oligos_count}')
print(f'final number of variants: {variants_count}')


# Produce number of active oligos in the entire hMPRA

In [ ]:
annotations = pd.read_csv(f'/home/labs/davidgo/Collaboration/humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv', 
                     header=0)
annotations = annotations.drop_duplicates(subset=["oligo"], keep = "first") #There are several HH oligos which are duplicated
annotations = annotations[#~(annotations['orientation_fix']=='fixed_in_L4')&
                           ~(annotations['oligo'].str.contains('SCREEN_EH'))&
                          ~(annotations['oligo'].str.contains('hh.missing.oligos')) ]


# create BED of annotations
annotations_bed_df = annotations[["chromosome","start","end","oligo"]]
annotations_bed_df.loc[:,"start"] = annotations_bed_df["start"].map(int) # already zero-based - Make sure with Simon!!
annotations_bed_df.loc[:,"end"] = annotations_bed_df["end"].map(int)
annotations_bed_df=annotations_bed_df.sort_values(['chromosome','start'], ascending = [True, True])
annotations_bed = pybedtools.BedTool.from_dataframe(annotations_bed_df)

# Number of sequences showing significant activity

In [ ]:
active_oligos_df = annotations.copy()
#active_oligos_count  = sum(active_oligos_df['activity_ancestral']=='active')+sum(active_oligos_df['activity_derived']=='active')
active_sequence_pair_counts = sum(~active_oligos_df['differential_activity'].isna())

print(f'Number of active sequences in the hMPRA:{active_sequence_pair_counts}')

print(f'percentage of active sequences in the hMPRA:{100*(active_sequence_pair_counts/oligo_pairs_count)}%')

# The number of observed differentially active oligos

In [ ]:
diff_active_oligos = annotations.copy()
diff_active_oligos = diff_active_oligos[diff_active_oligos['differential_activity']==True]
diff_active_oligos_count = len(diff_active_oligos)
print(f'Number of diff active oligos: {diff_active_oligos_count}')
print(f'Number of variants in diff active oligos: {sum(diff_active_oligos['variants_count'])}')
print(f'% of diff active oligo pairs out of all active oligo paris: {100*(diff_active_oligos_count/active_sequence_pair_counts)}')


# The number of positive controls in all libraries combined

In [ ]:
active_controls_chonds_ost = pd.read_excel(f'/home/labs/davidgo/Collaboration/humanMPRA_raw/sequences.xlsx', 
                    sheet_name = 'ctrl_active_chondr',
                     header=0)

active_controls_ESCs_Osteoblasts_NPCs = pd.read_excel(f'/home/labs/davidgo/Collaboration/humanMPRA_raw/sequences.xlsx', 
                    sheet_name = "ctrl_active_not_diff",
                     header=0)
active_controls_chonds_ost_L4 = pd.read_excel(
    "/home/labs/davidgo/Collaboration/humanMPRA_raw/L4/L4_v5.xlsx",
    sheet_name="ctrl_Pos_active_chondr",
    header=0)
active_controls_ESCs_Osteoblasts_NPCs_L4 = pd.read_excel(
    "/home/labs/davidgo/Collaboration/humanMPRA_raw/L4/L4_v5.xlsx",
    sheet_name="ctrl_Neg_diff",
    header=0)

In [ ]:
active_controls_count = len(active_controls_chonds_ost)+len(active_controls_ESCs_Osteoblasts_NPCs)+len(active_controls_chonds_ost_L4)+len(active_controls_ESCs_Osteoblasts_NPCs_L4)

print(f'Number of active controls in the library: {active_controls_count}')

# The number of negative controls in all libraries combined

In [ ]:
scrambled_control = pd.read_excel(f'/home/labs/davidgo/Collaboration/humanMPRA_raw/sequences.xlsx', 
                    sheet_name = 'ctrl_scrambled',
                     header=0)
NegCtrl_non_SCREEN = pd.read_excel(f'/home/labs/davidgo/Collaboration/humanMPRA_raw/sequences.xlsx', 
                    sheet_name = 'ctrl_non_SCREEN',
                     header=0)
NegCtrl_not_active_ESCs_Osteoblasts_NPCs = pd.read_excel(f'/home/labs/davidgo/Collaboration/humanMPRA_raw/sequences.xlsx', 
                    sheet_name = 'ctrl_not_active',
                     header=0)

scrambled_control_L4 = pd.read_excel(
    "/home/labs/davidgo/Collaboration/humanMPRA_raw/L4/L4_v5.xlsx", 
    sheet_name = "ctrl_scrambled",
    header=0)

# Note: there is no non-SCREEN controls in L4a1

NegCtrl_not_active_ESCs_Osteoblasts_NPCs_L4 = pd.read_excel(
    "/home/labs/davidgo/Collaboration/humanMPRA_raw/L4/L4_v5.xlsx",
    sheet_name="ctrl_Neg_active",
    header=0)

In [ ]:
neg_controls_count = len(scrambled_control)+len(NegCtrl_non_SCREEN)+len(NegCtrl_not_active_ESCs_Osteoblasts_NPCs)+len(scrambled_control_L4)+len(NegCtrl_not_active_ESCs_Osteoblasts_NPCs_L4)

print(f'Number of negative controls for activity in the library: {neg_controls_count}')

# Number of differentially active controls

In [ ]:
diff_activity_ctrls = pd.read_excel(f'/home/labs/davidgo/Collaboration/humanMPRA_raw/sequences.xlsx', 
                    sheet_name = 'ctrl_diff_active',
                     header=0)

diff_activity_ctrls_L4 = pd.read_excel("/home/labs/davidgo/Collaboration/humanMPRA_raw/L4/L4_v5.xlsx", 
    sheet_name = "ctrl_Pos_diff",
    header=0)


In [ ]:
diff_activity_ctrls

In [ ]:
diff_activity_ctrls_count = len(diff_activity_ctrls)+len(diff_activity_ctrls_L4)

print(f'Number of controls for differential activity in the library: {diff_activity_ctrls_count}')

# Total number of oligos in the library

In [ ]:
full_oligo_counts = diff_activity_ctrls_count+neg_controls_count+active_controls_count+oligos_count
print(f'Number of oligos in the library (+controls): {full_oligo_counts}')

# Top candidates information

In [ ]:
EVC = annotations[annotations['oligo']=='seq_9777_chr4:5774930-5775199_Hh+SCREEN_a3_L3']

## Find name of CSGALNACT1 eQTL

In [ ]:

R2Gtool = pd.read_csv(f'/home/labs/davidgo/Collaboration/humanMPRA/region_gene_link/chondrocytes/yardens_script/output_Nadav2_16-12-2024/Summary_PerEvidence.tsv', 
                     header=0,sep = '\t')

In [ ]:
R2Gtool[(R2Gtool['Gene_symbol']=='CSGALNACT1')&(R2Gtool['Distance_to_gene(TSS)']==3960)]

## Find expression LFC of CSGALNACT1

In [ ]:

G_hybrids = pd.read_csv(f'/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Hybrids/human_gorilla/ExpLBM/outputs_5Jan2026_for_hMPRA_draft1/ASE/combined_lfc_with_ASE_definition.tsv', 
                     header=0,sep = '\t')

C_hybrids = pd.read_csv(f"/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Hybrids/human_chimp/ExpLBM/outputs_05Jan2026_humanMPRA_draft1/ExpLBM_polarization_results.tsv", 
                     sep = '\t',
                     header=0)

In [ ]:
C_hybrids

In [ ]:
CGAT1_G_hybrids = G_hybrids[G_hybrids['Gene']=='CSGALNACT1']
CGAT1_C_hybrids = C_hybrids[C_hybrids['Gene']=='CSGALNACT1']

In [ ]:
CGAT1_G_hybrids

In [ ]:
CGAT1_C_hybrids

In [ ]:
import math

log2fc_value = -0.77425380391077
fold_change = 2**log2fc_value
percent_change = (fold_change - 1) * 100
print(f"{fold_change=}")
print(f"{percent_change=}")

In [ ]:
log2fc_value = -0.605798079693457
fold_change = 2**log2fc_value
percent_change = (fold_change - 1) * 100
print(f"{fold_change=}")
print(f"{percent_change=}")



# Create combined library design table (+L4)

In [ ]:
L123_library_design = pd.read_excel(f'/home/labs/davidgo/Collaboration/humanMPRA_raw/sequences.xlsx',
sheet_name='oligos')

L4_library_design = pd.read_excel(f'/home/labs/davidgo/Collaboration/humanMPRA_raw/L4/L4_v5.xlsx',
sheet_name='test_oligos')

In [ ]:
L4_library_design_filtered = L4_library_design[(L4_library_design['adapter']=='a1')&(L4_library_design['library']==('L4'))&(L4_library_design['class'].isin(['CGI.fix','promoter.orientation.fix']))&(L4_library_design['source'].str.contains('SCREEN'))]

In [ ]:
# Find rows in L123 that match L4 in chr, start, end
L123_L4_matches = L123_library_design.merge(
    L4_library_design_filtered[['chr', 'start', 'end']],
    on=['chr', 'start', 'end'],
    how='inner'
)

rows_removed = len(L123_L4_matches)

# Filter L123 to remove matching rows
L123_filtered = L123_library_design[
    ~L123_library_design.set_index(['chr', 'start', 'end']).index.isin(
        L4_library_design_filtered.set_index(['chr', 'start', 'end']).index
    )
].reset_index(drop=True)

print(f'Number of rows removed from L123: {rows_removed}')

In [ ]:
# Merge the filtered L4 and L123 datasets, keeping only shared columns (aligned + same order)
common_cols = L123_filtered.columns.intersection(L4_library_design_filtered.columns)
combined_library_design = pd.concat(
    [L4_library_design_filtered[common_cols], L123_filtered[common_cols]],
    ignore_index=True
)
combined_library_design = combined_library_design.drop_duplicates(keep="first")

#remove hedgehog only
combined_library_design = combined_library_design[combined_library_design['source'].str.contains('SCREEN')]


In [ ]:
all_variants = (
    combined_library_design["variants_genomic"]
      .dropna()
      .str.split(";")
      .explode()
      .tolist()
)
len(set(all_variants))

In [ ]:
# Create a publication-friendly version of the combined library design columns
publish_cols = {
    "chr": "Chromosome",
    "start": "Start",
    "end": "End",
    "variants_count": "Variants count",
    "variants_genomic": "Variants (genomic)",
    "variants_tile": "Variants (tile)",
    "adapter": "Adapter",
    "library": "Library",
    "ID": "ID",
    "source": "Source",
}

combined_library_design_pub = combined_library_design.rename(columns=publish_cols)

In [ ]:
len(combined_library_design_pub)*2


In [ ]:
combined_library_design_pub.to_excel(f'/home/labs/davidgo/Collaboration/humanMPRA/paper/extended_datasets/combined_library_design.xlsx', index=False)

# Compare library design to final oligo output

In [ ]:
library_design = pd.read_excel('/home/labs/davidgo/Collaboration/humanMPRA/paper/extended_datasets/combined_library_design.xlsx')

In [ ]:
final_output = pd.read_csv("/home/labs/davidgo/Collaboration/humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv")

In [ ]:
final_output_oligos_id = final_output["oligo"].str.extract(r"^seq_(\d+)_", expand=False)
final_output_oligos_id = pd.to_numeric(final_output_oligos_id, errors="coerce").astype("Int64")


In [ ]:
library_design_oligos_id = library_design["ID"].astype("int64")


In [ ]:
missing_in_output = library_design_oligos_id[~library_design_oligos_id.isin(final_output_oligos_id)]

In [ ]:
# Make sure both sides are int
ids = pd.to_numeric(library_design["ID"], errors="coerce").astype("Int64")
missing = pd.to_numeric(missing_in_output, errors="coerce").astype("Int64")

missing_oligos = library_design[ids.isin(missing)]
detected_oligos = library_design[~ids.isin(missing)]

In [ ]:
detected_oligos_L4a1 = detected_oligos[(detected_oligos['Library']=='L4') & (detected_oligos['Adapter']=='a1')]
missing_oligos_L4a1 = missing_oligos[(missing_oligos['Library']=='L4') & (missing_oligos['Adapter']=='a1')]


In [ ]:
missing_oligos_L4a1

In [ ]:
# search for missing L4 a1 oligos in the barcode association file
csv_path = "/home/labs/davidgo/Collaboration/humanMPRA/neurons/L4a1/output/DNA_barcode_associations_2/Omer_re-send/final_df.csv"

# read (no header expected based on example rows)
cols = ["barcode_id", "barcode_seq", "oligo", "count"]
assoc = pd.read_csv(csv_path, header=None, names=cols, dtype=str)

# extract numeric oligo ID from the oligo string like "seq_255337_..."
assoc["oligo_id"] = assoc["oligo"].str.extract(r"^seq_(\d+)_", expand=False).astype("Int64")

# prepare list of missing IDs (Int64)
missing_ids = missing_oligos_L4a1["ID"].astype("Int64").unique()

# filter associations for missing IDs and aggregate per oligo
matches = assoc[assoc["oligo_id"].isin(missing_ids)].copy()
agg = (
    matches.groupby("oligo_id")
    .agg(
        n_rows=("oligo", "size"),
        n_unique_barcodes=("barcode_seq", "nunique"),
        barcodes=("barcode_seq", lambda x: ";".join(x.unique()))
    )
    .reset_index()
    .rename(columns={"oligo_id": "ID"})
)

# ensure all missing IDs are present in the output
out = pd.DataFrame({"ID": pd.Series(missing_ids, dtype="Int64")}).merge(agg, on="ID", how="left")
out[["n_rows", "n_unique_barcodes"]] = out[["n_rows", "n_unique_barcodes"]].fillna(0).astype(int)
out["barcodes"] = out["barcodes"].fillna("")

# save results and show a brief summary
out.to_csv("/home/labs/davidgo/Collaboration/humanMPRA/paper/extended_datasets/missing_L4a1_barcode_hits.csv", index=False)
print(f"Missing IDs checked: {len(out)}; IDs with >=1 hit: {int((out['n_rows']>0).sum())}")

In [ ]:
library_design
plt.hist(library_design['ID'],bins=100)

In [ ]:
plt.hist(missing_in_library['ID'],bins=100)

In [ ]:
print(missing_in_library[missing_in_library['Variants (genomic)']=='chr10:18872538(G->A);chr10:18872581(T->C);chr10:18872697(C->T)'])

In [ ]:
contained_in_library = library_design[~ids.isin(missing)]


In [ ]:
contained_in_library[contained_in_library['Library']=='L3']

## Create Activity supplementary table

In [ ]:
#too heavy to hold everything in memory, so I will delete the dataframes I don't need anymore
del [library, library_L4, library_L4_filtered]


In [ ]:
full_activity_df = pd.read_csv('/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/quantitative_analysis_combined/comb_df_combined_fdr.csv')


In [ ]:
#remove unnecesary columns to save memory (everything except RNA_chondrocytes_rep1)
full_activity_df = full_activity_df.drop(columns=['RNA_chondrocytes_rep1', 'RNA_chondrocytes_rep2', 'RNA_chondrocytes_rep3',
                                                    'DNA_chondrocytes_rep1', 'DNA_chondrocytes_rep2', 'DNA_chondrocytes_rep3'])

In [ ]:
#filter the df to contain only oligos which have their names in final_oligos['name_ancestral']or final_oligos['name_derived']. For oligos in final oligos derived or ancestral that are not present in the activity table,  Add NAs for the activity columns.
full_activity_pub = full_activity_df[full_activity_df['oligo'].isin(final_oligos['name_ancestral'].tolist() + final_oligos['name_derived'].tolist())]

# All oligos that should appear in the final table
expected_oligos = pd.concat([
    final_oligos['name_ancestral'],
    final_oligos['name_derived']
]).drop_duplicates().to_frame(name='oligo')

# Keep all expected oligos.
# If an oligo is missing from full_activity_df, its activity columns become NA.
full_activity_pub = expected_oligos.merge(
    full_activity_df,
    on='oligo',
    how='left'
)

full_activity_pub['coords'] = full_activity_pub['oligo'].str.extract(r'(chr[\w]+:\d+-\d+)')

#full_activity_pub = full_activity_pub[[ 'oligo', 'coords', 'mad.score','count_rep_comb','RNA_rep_comb','DNA_rep_comb','ratio_log_rep_comb','pval.mad','fdr.mad_adjusted_combined','activity_adjusted_combined']]

#Make classification pretty using dict
activity_mapping = {
    'non_active': 'Not active',
    'active': 'Active',
    np.nan: np.nan
}
full_activity_pub['activity_adjusted_combined'] = full_activity_pub['activity_adjusted_combined'].map(activity_mapping)

full_activity_pub.to_csv("/home/labs/davidgo/Collaboration/humanMPRA/paper/extended_datasets/activity_table.csv", index=False)


In [ ]:
full_activity_pub

## Activity of controls

In [ ]:
full_activity_df_ctrls = full_activity_df[~(full_activity_df['control_type'].str.contains('No control') |
                                            #(full_activity_df['control_type'].str.contains('NegCtrl_active_not_diff') |
                                             full_activity_df['control_type'].str.contains('PosCtrl_neuron_active'))]




In [ ]:
#dict to change contro_type
control_type_mapping = {
    'NegCtrl_active_not_diff': 'Positive control (active)',
    'ctrl_Neg_diff': 'Positive control (active)',
    'ctrl_diff_active': 'Positive control (differential activity)',
    'ctrl_Neg_active': 'Negative control (not active)',
    'ctrl_scrambled': 'Negative control (scrambled)',
    'ctrl_non_SCREEN': 'Negative control (non-SCREEN)',
    np.nan: np.nan
}

#apply
full_activity_df_ctrls['control_type'] = full_activity_df_ctrls['control_type'].map(control_type_mapping)

full_activity_df_ctrls.to_csv("/home/labs/davidgo/Collaboration/humanMPRA/paper/extended_datasets/activity_table_ctrls.csv", index=False)


In [ ]:
full_activity_df_ctrls['control_type'].unique()

In [ ]:
full_activity_df_ctrls